In [1]:
%load_ext autoreload

In [2]:

import pandas as pd
import json
import sklearn
import glob
import pickle
from sklearn.model_selection import train_test_split
from collections import Counter


pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
%autoreload
import sys
sys.path.insert(0, '../../style_generation_pipeline')

from data import *
from cluster_representation import *

/mnt/swordfish-pool2/milad/conda-envs/gpu-env/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2025-03-06 08:30:38.549721: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741267838.564053  123560 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741267838.568164  123560 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-06 08:30:38.582899: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimize

In [49]:
path='/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/deepseek_style_experiment/'

In [46]:
all_documents = pd.read_json(path + 'all_document.jsonl', lines=True)

In [80]:
#raw_style_feats = pd.read_pickle(path + '/describe_documents_writing_styles_raw_generations.pkl')

In [83]:
style_feats_df = pd.read_csv(path + '/filtered/refined_and_aggregated_features_final.csv')
feat_to_ling_lvl = json.load(open(path + '/feats_to_ling_lvl.json'))
style_feats_df['ling_lvl'] = style_feats_df.original_attribute_name.apply(lambda x: feat_to_ling_lvl[x] if x in feat_to_ling_lvl else 'other')

In [82]:
style_feats_df.documentID.nunique()

22701

In [84]:
style_feats_df.head()

,final_attribute_name,original_attribute_name,documentID,shortend_attribute_name.v1,shortend_attribute_name.v2,aggregated_name,ling_lvl
0,other,other,000206ff-14aa-533c-8193-2a02f438cb57,other,other,other,Syntactic Level
1,other,other,000206ff-14aa-533c-8193-2a02f438cb57,other,other,other,Syntactic Level
2,other,other,000206ff-14aa-533c-8193-2a02f438cb57,other,other,other,Syntactic Level
3,other,other,000206ff-14aa-533c-8193-2a02f438cb57,other,other,other,Syntactic Level
4,other,other,000bed09-eaef-53b5-b9c1-f46ada93e3d1,other,other,other,Syntactic Level


In [87]:
style_feats_df.final_attribute_name.value_counts()

other                                                                                 74602
generic sentence                                                                       6828
passive voice                                                                          1607
varied sentence structures                                                             1310
technical terms                                                                         826
specific terms                                                                          749
complex sentence structures                                                             692
specific terminology                                                                    658
The author employs a variety of sentence structures.                                    571
grammatical structures                                                                  526
different sentence structures                                                   

In [88]:
style_feats_df.documentID.value_counts()

7952c2d0-e004-5b02-87a6-386265a7e395    58
61be0dcf-3f51-5c34-9c6c-036ee7f11301    54
63bd70b8-5199-5c3e-9fe8-42053f0e6768    50
e55b99fe-5491-584c-920e-638a05eb5735    48
07c9728c-cd33-515a-b760-621d8b6537d2    48
73ce1574-69f1-5f1b-9f2c-c3d65a7b72f5    44
dfe618c3-b0c7-5fd2-84bb-595e3c20f898    40
840363ca-1828-52f0-ab29-08e7979dfe48    40
7077d70e-a38a-5b12-a828-04d08a29b6dd    39
fb5ecf6c-67de-5dc3-aeda-a0ad3e99a357    36
123a1e02-1e9d-50d5-a07e-edd5ac6c1623    35
76edf775-f3ac-5144-9c9c-61de11d44d8f    35
6f899f16-bf99-54b3-8096-eaf577952cde    35
301e6885-b43e-53a7-9fe6-586f110c4c05    34
545d9c44-7f00-58dd-854a-0d0026784e45    33
45d1c76d-523a-5b6a-9c72-ac18c5301cce    33
ca30a4b5-dba2-569c-a44e-e6f1d41d35ff    33
f2caae33-d0a4-57d7-9ffb-b22b8772160b    33
083710df-bdc7-5bf5-b324-393e8970b662    31
22cf67aa-d3ed-5d2c-8c29-bed599e1bacd    31
b0451355-9993-5ee4-9cf4-cd6cef20f3ad    31
a607bc70-b71e-5199-a97e-52aec6fcd949    31
52cdc260-b305-577f-9ee1-eb2a7f70c56e    31
4a605ce2-94

In [89]:
print(style_feats_df.documentID.nunique())
print(style_feats_df['shortend_attribute_name.v2'].nunique(), style_feats_df.aggregated_name.nunique(), style_feats_df.final_attribute_name.nunique())
print(style_feats_df.ling_lvl.nunique())

22701
2187 2156 1069
5


In [90]:
style_feats_df.to_csv(path + '/refined_and_aggregated_features_final.csv', index=False)

In [91]:
style_feats_df.groupby(['final_attribute_name', 'aggregated_name']).agg({'documentID': lambda x: len(x), 'ling_lvl': lambda x: list(x)[0]}).reset_index().to_csv(path + '/llm_generated_style_feats.csv')

In [97]:
g_df = style_feats_df.groupby('aggregated_name').agg({'original_attribute_name': lambda feats: {f: feats.tolist().count(f) for f in set(feats)},
                                               'shortend_attribute_name.v1': lambda feats: {f: feats.tolist().count(f) for f in set(feats)},
                                               'shortend_attribute_name.v2': lambda feats: {f: feats.tolist().count(f) for f in set(feats)},
                                               'documentID': lambda x: len(x),
                                               'ling_lvl': lambda x: list(x)
                                    }).reset_index()

g_df['final_attribute_ling_lvl'] = g_df['ling_lvl'].apply(lambda x: Counter(x).most_common(1)[0][0])

In [98]:
g_df.to_json(path + '/style_features_corpus.json', orient='records', indent=2)

In [99]:
g_df.final_attribute_ling_lvl.value_counts()

Morphological Level    778
Syntactic Level        653
Discourse Level        413
Semantic Level         312
Name: final_attribute_ling_lvl, dtype: int64